# Parsers against each other and against the judge

Runs the current Python parser, the Scala `PeriodParser` (through `PeriodParserCli` and sbt) and
`period_new` over every string in `data/periods.jsonl`, shows where they disagree, and scores each
against the judge's answers in `data/judge-output`. Ranges are compared as inclusive day bounds
with open sides as `None`; identifiers are not compared.

In [1]:
import json
import subprocess
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

from adapters.transformers.marc.parsers.period import parse as parse_new

rows = [json.loads(line) for line in Path("data/periods.jsonl").open(encoding="utf-8")]
occurrences = Counter()
for row in rows:
    occurrences[(row["source"], row["path"], row["text"])] += row["n"]
print(f"{sum(occurrences.values())} strings, {len(occurrences)} distinct (source, path, text) triples")

1547871 strings, 76343 distinct (source, path, text) triples


In [2]:
OPEN = {"0001-01-01", "9999-12-31", "-9999-01-01"}


def bounds(start, end):
    return tuple(None if not d or d[:10] in OPEN else d[:10] for d in (start, end))


def new_bounds(source, path, text):
    span = parse_new(text, source)
    return bounds(span[0].isoformat(), span[1].isoformat()) if span else (None, None)

In [3]:
scala_in = Path("data/scala-in.txt").resolve()
scala_out = Path("data/scala-out.jsonl").resolve()
scala_in.write_text("\n".join(sorted({text.replace("\n", " ") for _, _, text in occurrences})), encoding="utf-8")
sbt = subprocess.run(
    ["sbt", "-batch", f"transformer_common/Test/runMain weco.pipeline.transformer.parse.PeriodParserCli {scala_in} {scala_out}"],
    cwd="../..", capture_output=True, text=True,
)
assert sbt.returncode == 0, sbt.stdout[-2000:] + sbt.stderr[-2000:]

scala = {}
for line in scala_out.read_text(encoding="utf-8").splitlines():
    row = json.loads(line)
    rng = row["range"] or {}
    scala[row["input"]] = bounds(rng.get("from"), rng.get("to"))

PARSERS = {"scala": lambda source, path, text: scala[text], "new": new_bounds}
results = {name: {key: parser(*key) for key in occurrences} for name, parser in PARSERS.items()}

NameError: name 'python_bounds' is not defined

In [4]:
def verdict(key):
    got = {name: results[name][key] for name in PARSERS}
    if len(set(got.values())) == 1:
        return "all agree"
    return "differ: " + ", ".join(sorted(name for name in PARSERS if list(got.values()).count(got[name]) == 1))


verdicts = {key: verdict(key) for key in occurrences}
summary = Counter()
for key, v in verdicts.items():
    summary[(key[1], v)] += occurrences[key]
for (path, v), n in sorted(summary.items()):
    print(f"{path:11} {v:24} {n:>9}")

NameError: name 'PARSERS' is not defined

In [5]:
LIMIT = 40


def show(b):
    return f"{b[0] or 'open'} .. {b[1] or 'open'}" if b != (None, None) else "no range"


for key in sorted((k for k in occurrences if verdicts[k] != "all agree"), key=lambda k: -occurrences[k])[:LIMIT]:
    source, path, text = key
    print(f"{text!r}  [{source} {path}, {occurrences[key]}x]")
    for name in PARSERS:
        print(f"    {name:7} {show(results[name][key])}")

NameError: name 'verdicts' is not defined

## Against the judge

`unparseable` and `ambiguous` both mean no range is right. Judge answers that are malformed or
run backwards are dropped. Precision is correct ranges over ranges produced; recall is correct
ranges over ranges the judge found.

In [6]:
index = json.load(Path("data/judge-input/index.json").open(encoding="utf-8"))
judged, malformed = {}, 0
for f in sorted(Path("data/judge-output").glob("batch-*.jsonl")):
    for line in f.read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.startswith("```"):
            continue
        try:
            id_, *rest = json.loads(line)
            assert len(rest) == 5
            judged[id_] = tuple(rest)
        except (ValueError, AssertionError):
            malformed += 1


def valid(outcome, start, end):
    if outcome != "range":
        return outcome in ("unparseable", "ambiguous")
    try:
        return bool(start or end) and all(date.fromisoformat(d.lstrip("-")) for d in (start, end) if d) and (not start or not end or start <= end)
    except ValueError:
        return False


expected = {}
for id_, (outcome, start, end, qualifier, note) in judged.items():
    if valid(outcome, start, end):
        for path in index[id_]["paths"]:
            expected[(id_, path)] = bounds(start, end) if outcome == "range" else (None, None)

got = {name: {(id_, path): results[name][(index[id_]["source"], path, index[id_]["text"])] for id_, path in expected} for name in PARSERS}
weight = lambda key: occurrences[(index[key[0]]["source"], key[1], index[key[0]]["text"])]

print(f"{len(judged)} judged, {malformed} malformed lines skipped, {len(judged) - len({k[0] for k in expected})} dropped as invalid\n")
print(f"{'parser':8} {'correct':>8} {'wrong':>6}   {'accuracy':>9} {'precision':>10} {'recall':>7}   {'weighted acc':>12} {'prec':>6} {'recall':>7}")
for name in PARSERS:
    line = f"{name:8}"
    for w in (lambda key: 1, weight):
        total = sum(w(k) for k in expected)
        correct = sum(w(k) for k in expected if got[name][k] == expected[k])
        produced = sum(w(k) for k in expected if got[name][k] != (None, None))
        judge_ranges = sum(w(k) for k in expected if expected[k] != (None, None))
        correct_ranges = sum(w(k) for k in expected if got[name][k] == expected[k] != (None, None))
        cells = (correct / total, correct_ranges / produced, correct_ranges / judge_ranges)
        line += f" {correct:>8} {total - correct:>6}   {cells[0]:9.1%} {cells[1]:10.1%} {cells[2]:7.1%}" if w(next(iter(expected))) == 1 and w is not weight else f"   {cells[0]:12.1%} {cells[1]:6.1%} {cells[2]:7.1%}"
    print(line)

NameError: name 'PARSERS' is not defined

In [7]:
PARSER = "new"
LIMIT = 40

for key in sorted((k for k in expected if got[PARSER][k] != expected[k]), key=lambda k: -weight(k))[:LIMIT]:
    id_, path = key
    outcome, start, end, qualifier, note = judged[id_]
    print(f"{index[id_]['text']!r}  [{path}, {weight(key)}x]")
    print(f"    judge   {show(expected[key]) if outcome == 'range' else outcome}{'  ' + repr(qualifier) if qualifier and qualifier != 'exact' else ''}{'  ' + note if note else ''}")
    print(f"    {PARSER:7} {show(got[PARSER][key])}")

## Against the 008

Sierra production dates paired with the record's own 008 range (from `period_sierra_extract`), compared
at year level since the 008 has no months. Only strings the parsers saw in `periods.jsonl` are scored.

In [8]:
def bounds_008(value):
    if value is None:
        return None
    start, sep, end = value.partition("-")
    return (start or None, (end or None) if sep else start)


def years_of(b):
    return None if b == (None, None) else tuple(d and d[:4] for d in b)


pairs = [json.loads(line) for line in Path("data/sierra-008.jsonl").open(encoding="utf-8")]
scored = Counter()
mismatches = defaultdict(Counter)
for row in pairs:
    key = ("marc", "production", row["text"])
    gold = bounds_008(row["range_008"])
    if gold is None or key not in occurrences:
        continue
    for name in PARSERS:
        got = years_of(results[name][key])
        outcome = "agree" if got == gold else "no range" if got is None else "disagree"
        scored[(name, outcome)] += row["n"]
        if outcome != "agree":
            mismatches[name][(row["text"], row["range_008"], show(results[name][key]))] += row["n"]

total = sum(n for (name, _), n in scored.items() if name == "new")
print(f"{total} production strings with an 008 range\n")
print(f"{'parser':8} {'agree':>8} {'no range':>9} {'disagree':>9}")
for name in PARSERS:
    print(f"{name:8} {scored[(name, 'agree')] / total:8.1%} {scored[(name, 'no range')] / total:9.1%} {scored[(name, 'disagree')] / total:9.1%}")

print("\nnew parser vs 008, heaviest mismatches")
for (text, range_008, got), n in mismatches["new"].most_common(30):
    print(f"{n:6}  {text!r:40} 008 {range_008:12} new {got}")

NameError: name 'PARSERS' is not defined